# Train C51

C51 learns a categorical return distribution over 51 fixed atoms and projects each Bellman target back onto that support. This notebook trains it on `CartPole-v1` and plots its learning curve.

## Defining idea

$$\mathcal L=-\sum_i(\Phi T Z)_i\log p_i(s,a).$$

Here $p_i(s,a)$ is the predicted probability of atom $i$, $TZ$ is the Bellman-updated return distribution, and $\Phi$ projects it onto the fixed support.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import C51, C51Config
from aprenderl.utils import evaluate_policy

ENV_ID = "CartPole-v1"

In [ ]:
env = gym.make(ENV_ID)
config = C51Config(
    atoms=51,
    v_min=-10.0,
    v_max=10.0,
    buffer_size=10_000,
    batch_size=64,
    learning_starts=500,
    target_update_interval=250,
    exploration_steps=2_000,
    seed=7,
)

agent = C51(env, config=config, device="cpu")
agent.learn(total_timesteps=5_000, progress_bar=False)
env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(
    returns, np.ones(window) / window, mode="valid"
)

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"C51 training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:

evaluation_env = gym.make(ENV_ID, render_mode="human")
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")